In [ ]:
#cell 1
!apt-get install -y default-jdk -q
!java -version

In [ ]:
# Cell 2: Download CloudSimPlus
!wget -q https://repo1.maven.org/maven2/org/cloudsimplus/cloudsim-plus/6.3.0/cloudsim-plus-6.3.0.jar
!wget -q https://repo1.maven.org/maven2/org/apache/commons/commons-math3/3.6.1/commons-math3-3.6.1.jar
!wget -q https://repo1.maven.org/maven2/org/slf4j/slf4j-api/1.7.36/slf4j-api-1.7.36.jar
!wget -q https://repo1.maven.org/maven2/org/slf4j/slf4j-simple/1.7.36/slf4j-simple-1.7.36.jar
!wget -q https://repo1.maven.org/maven2/org/apache/commons/commons-lang3/3.12.0/commons-lang3-3.12.0.jar
!ls -lh *.jar

In [ ]:
#cell 3
%%writefile VolunteerFogTraceGenerator.java

import org.cloudbus.cloudsim.core.CloudSim;
import org.apache.commons.math3.distribution.*;
import org.apache.commons.math3.random.JDKRandomGenerator;
import org.apache.commons.math3.random.RandomGenerator;

import java.io.FileWriter;
import java.io.IOException;
import java.util.ArrayList;
import java.util.List;
import java.util.Random;

public class VolunteerFogTraceGenerator {

    private static final int    NUM_NODES        = 2500;
    private static final double TICK_INTERVAL    = 60.0;
    private static final double SIMULATION_LIMIT = 3600.0;
    private static final String OUTPUT_FILE      = "volunteer_fog_dataset.csv";
    private static final long   SEED             = 42L;

    private static final int EVENT_TICK = 1;

    public static void main(String[] args) {
        Random rng = new Random(SEED);
        RandomGenerator rg = new JDKRandomGenerator();
        rg.setSeed(SEED);

        // Updated distributions
        GammaDistribution     rtDist    = new GammaDistribution(rg, 1.2, 200.0);
        BetaDistribution      availDist = new BetaDistribution(rg, 5.0, 2.0);
        LogNormalDistribution tpDist    = new LogNormalDistribution(rg, 3.5, 0.6);

        CloudSim simulation = new CloudSim();

        List<VolunteerNode> nodePool = new ArrayList<>();
        for (int i = 1; i <= NUM_NODES; i++) {
            double baseAvail = availDist.sample() * 100.0;
            nodePool.add(new VolunteerNode(i, baseAvail, rtDist, tpDist));
        }

        TraceDriverEntity traceDriver = new TraceDriverEntity(
            "TraceDriver", simulation, nodePool, rng,
            SIMULATION_LIMIT, TICK_INTERVAL, EVENT_TICK);

        System.out.println("Starting CloudSimPlus trace simulation...");
        System.out.println("Seed:     " + SEED);
        System.out.println("Nodes:    " + NUM_NODES);
        System.out.println("Duration: " + SIMULATION_LIMIT + " seconds");

        simulation.start();
        exportResults(nodePool, rng);
        System.out.println("Dataset exported to: " + OUTPUT_FILE);
    }

    private static void exportResults(List<VolunteerNode> nodes, Random rng) {
        try (FileWriter writer = new FileWriter(OUTPUT_FILE)) {
            writer.append("node_id,Response_Time,Availability,Throughput,Reliability,Latency\n");
            for (VolunteerNode n : nodes) {
                n.calculateFinalMetrics(rng);
                writer.append(String.format("%d,%.2f,%.2f,%.3f,%.2f,%.2f\n",
                        n.id, n.avgRT, n.avgAvail, n.avgTP, n.derivedRel, n.derivedLat));
            }
        } catch (IOException e) {
            System.err.println("Error writing CSV: " + e.getMessage());
        }
    }

    static class VolunteerNode {
        int id;
        boolean isOnline = true;
        double baseTargetAvail;
        GammaDistribution rtSampler;
        LogNormalDistribution tpSampler;

        int    totalTicks   = 0;
        int    upTicks      = 0;
        double cumulativeRT = 0;
        double cumulativeTP = 0;

        double avgRT, avgAvail, avgTP, derivedRel, derivedLat;

        VolunteerNode(int id, double baseAvail, GammaDistribution rt, LogNormalDistribution tp) {
            this.id              = id;
            this.baseTargetAvail = baseAvail;
            this.rtSampler       = rt;
            this.tpSampler       = tp;
        }

        void recordTickTrace(Random rng) {
            totalTicks++;

            // Personalised Markov probabilities from Beta availability
            double pStayUp  = Math.min(0.99, 0.7 + (baseTargetAvail / 300.0));
            double pRecover = Math.min(0.50, 0.1 + (baseTargetAvail / 500.0));

            double rand = rng.nextDouble();
            isOnline = isOnline ? rand < pStayUp : rand < pRecover;

            if (isOnline) {
                upTicks++;
                cumulativeRT += rtSampler.sample();
                cumulativeTP += tpSampler.sample();
            } else {
                cumulativeTP += 0.005;
            }
        }

        void calculateFinalMetrics(Random rng) {
            this.avgAvail   = ((double) upTicks / totalTicks) * 100.0;
            this.avgRT      = upTicks > 0 ? (cumulativeRT / upTicks) : 2000.0;
            this.avgTP      = cumulativeTP / totalTicks;
            this.derivedRel = Math.min(100.0, (avgAvail * 0.96) + (rng.nextDouble() * 3));
            this.derivedLat = (avgRT * 0.38) + (rng.nextDouble() * 15);
        }
    }
}

In [ ]:
#cell 4
%%writefile TraceDriverEntity.java
import org.cloudbus.cloudsim.core.CloudSimEntity;
import org.cloudbus.cloudsim.core.Simulation;
import org.cloudbus.cloudsim.core.events.SimEvent;
import java.util.List;
import java.util.Random;

public class TraceDriverEntity extends CloudSimEntity {
    private final List<VolunteerFogTraceGenerator.VolunteerNode> nodePool;
    private final Random rng;
    private final double simulationLimit;
    private final double tickInterval;
    private final int eventTick;

    public TraceDriverEntity(String name, Simulation simulation,
                         List<VolunteerFogTraceGenerator.VolunteerNode> nodePool,
                         Random rng, double simulationLimit,
                         double tickInterval, int eventTick) {
    super(simulation);
    setName(name);
    this.nodePool        = nodePool;
    this.rng             = rng;
    this.simulationLimit = simulationLimit;
    this.tickInterval    = tickInterval;
    this.eventTick       = eventTick;
    }

    @Override
    protected void startInternal() {
        schedule(this, tickInterval, eventTick);
    }

    @Override
    public void processEvent(SimEvent ev) {
        if (ev.getTag() == eventTick) {
            double time = getSimulation().clock();
            if (time < simulationLimit) {
                nodePool.forEach(n -> n.recordTickTrace(rng));
                schedule(this, tickInterval, eventTick);
            } else {
                System.out.println("Simulation limit reached. Stopping TraceDriver.");
            }
        }
    }
}

In [ ]:
#cell 5
!javac -cp "cloudsim-plus-6.3.0.jar:commons-math3-3.6.1.jar:slf4j-api-1.7.36.jar:slf4j-simple-1.7.36.jar:commons-lang3-3.12.0.jar" VolunteerFogTraceGenerator.java TraceDriverEntity.java

In [ ]:
#cell 6
!java -cp ".:cloudsim-plus-6.3.0.jar:commons-math3-3.6.1.jar:slf4j-api-1.7.36.jar:slf4j-simple-1.7.36.jar:commons-lang3-3.12.0.jar" VolunteerFogTraceGenerator

In [ ]:
#cell 7
import pandas as pd
df = pd.read_csv('volunteer_fog_dataset.csv')
print(f'Rows: {len(df)}')
print(df.describe())

In [ ]:
#cell 8
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy(
    'volunteer_fog_dataset.csv',
    '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/QWS to VFBC synthetic transferability/Synthetic_VBFC_Dataset/synthetic_vbfc_dataset.csv'
)